<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Appendix D: Using larger LLMs

Packages that are being used in this notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- The main chapters use the Qwen3 0.6B base model because it is the smallest model in the
Qwen3 family and therefore the easiest to run on consumer hardware
- However, the same `Qwen3Model` implementation from appendix C can also be used to load larger dense Qwen3 checkpoints with the same from-scratch PyTorch model code

&nbsp;
## D.1 Larger dense Qwen3 configurations

The repository includes configuration dictionaries for several larger dense Qwen3 models (beyond the 0.6B model) in
`reasoning_from_scratch.appendix_c` ([reasoning_from_scratch/appendix_c.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/appendix_c.py)):

| Model size | Configuration dictionary |
| --- | --- |
| 1.7B | `QWEN3_CONFIG_1_7B` |
| 4B | `QWEN3_CONFIG_4B` |
| 8B | `QWEN3_CONFIG_8B` |
| 14B | `QWEN3_CONFIG_14B` |
| 32B | `QWEN3_CONFIG_32B` |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-d/Appendix_D_F01_raschka.webp" width="500px">

- As mentioned in the figure above, these are the "dense" Qwen3 variants, which can run on single GPUs
- There are also "sparse" Mixture-of-Experts variants of Qwen3, but they are not supported via this books' code; however, if you are interested in a from-scratch implementation, you can find one here: https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/11_qwen3
- All of these use the same overall architecture pattern as the 0.6B model from appendix C
- What changes are the embedding size, number of layers, number of attention heads, and
feed-forward hidden dimension

- As a rough lower bound, storing weights in bfloat16 requires about 2 bytes per parameter
- This means that the checkpoint weights alone are on the order of:

| Model size | Rough weight memory in bfloat16 |
| --- | --- |
| 1.7B | about 3.4 GB |
| 4B | about 8 GB |
| 8B | about 16 GB |
| 14B | about 28 GB |
| 32B | about 64 GB |


- In practice, the real runtime memory usage is higher because we also need memory for
activations, temporary buffers, and often the KV cache

&nbsp;
## D.2 Downloading larger checkpoints overview

- Unlike the 0.6B checkpoints used in the main chapters, larger official Qwen3 models are
typically distributed as `safetensors` files, sometimes split across multiple shards
- The helper function `download_from_huggingface_from_snapshots` to load these requires some additional packages:

```bash
!uv add huggingface_hub safetensors
```

or

```bash
!pip install huggingface_hub safetensors
```

&nbsp;
## D.3 Loading a larger base model

- Download weights:

In [2]:
from pathlib import Path
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.appendix_c import (
    download_from_huggingface_from_snapshots
)


device = get_device()
local_dir = Path("qwen3-4b-base")

weights = download_from_huggingface_from_snapshots(
    repo_id="Qwen/Qwen3-4B-Base",
    local_dir=local_dir,
)

Using Apple Silicon GPU (MPS)


/Users/sebastian/Developer/reasoning-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 2616.79it/s]


- Initialize model:

In [3]:
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, load_hf_weights_into_qwen
)
from reasoning_from_scratch.appendix_c import QWEN3_CONFIG_4B


model = Qwen3Model(QWEN3_CONFIG_4B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN3_CONFIG_4B["n_layers"],
        "hidden_dim": QWEN3_CONFIG_4B["hidden_dim"],
    },
    params=weights,
)
model.to(device)
model.eval()

Model uses weight tying.


Qwen3Model(
  (tok_emb): Embedding(151936, 2560)
  (trf_blocks): ModuleList(
    (0-35): 36 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2560, out_features=4096, bias=False)
        (W_key): Linear(in_features=2560, out_features=1024, bias=False)
        (W_value): Linear(in_features=2560, out_features=1024, bias=False)
        (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=2560, out_features=9728, bias=False)
        (fc2): Linear(in_features=2560, out_features=9728, bias=False)
        (fc3): Linear(in_features=9728, out_features=2560, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=2560, out_features=151936, bias=False)
)

- Load tokenizer:

In [4]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
import shutil

# Note that the original base tokenizer is called "tokenizer.json"
# We rename it to distinguish from the reasoning tokenizer (next section)
tokenizer_src = local_dir / "tokenizer.json"
tokenizer_path = local_dir / "tokenizer-base.json"

if not tokenizer_path.exists():
    shutil.copyfile(tokenizer_src, tokenizer_path)

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- Use model:

In [5]:
import torch
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache,
)

prompt = "Explain large language models in two sentences."
input_ids = torch.tensor(
    tokenizer.encode(prompt),
    device=device,
).unsqueeze(0)

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=64,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)

 Large language models are artificial intelligence systems that use deep learning techniques to understand and generate human-like text. They are trained on vast amounts of data and can perform a wide range of natural language processing tasks, such as translation, summarization, and question answering.

&nbsp;
## D.4 Loading a larger reasoning variant

- The same idea also works for larger reasoning-style Qwen3 models
- The architecture for a given model size stays the same; only the checkpoint and tokenizer settings change

For example, to load the 4B reasoning variant instead of the 4B base variant, we would:

- switch the repository ID from `Qwen/Qwen3-4B-Base` to `Qwen/Qwen3-4B`;
- copy the `tokenizer.json` file to `tokenizer-reasoning.json`;
- initialize the tokenizer as follows:

```python
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_path,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
```

- The rest of the model-loading and -usage code stays the same

&nbsp;
## D.5 Practical recommendations

- No code in this section